In [37]:
import os
import ssl
import urllib.request

# 1. Force SSL context to be unverified
ssl._create_default_https_context = ssl._create_unverified_context

# 2. Tell OpenMP (used by DocTR/PyTorch) it's okay to run in a "fragile" environment
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

In [38]:
# import libraries

import pandas as pd
import json
import requests
from doctr.models import ocr_predictor
from doctr.io import DocumentFile

In [39]:
# This tells the script to ignore SSL certificate verification
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

In [40]:
# 1. OCR: Extract text using DocTR
def extract_text_doctr(image_path):
    model = ocr_predictor(det_arch='db_resnet50', reco_arch='crnn_vgg16_bn', pretrained=True)
    doc = DocumentFile.from_images([image_path])
    result = model(doc)
    
    # Export to structured text (maintaining basic order)
    full_text = ""
    for page in result.pages:
        for block in page.blocks:
            for line in block.lines:
                line_text = " ".join([word.value for word in line.words])
                full_text += line_text + "\n"
    return full_text

In [41]:
# 2. EXTRACTION: Send to local Ollama (BioMistral or Llama-3)
def extract_with_local_llm(raw_text):
    prompt = f"""
    Extract data from this medical bill OCR text. 
    Respond ONLY with valid JSON.
    Fields: patient_name, provider_name, date, total_due, billing_codes (list).
    
    TEXT:
    {raw_text}
    """
    
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "cniongolo/biomistral", 
            "prompt": prompt,
            "stream": False,
            "format": "json"
        }
    )
    return json.loads(response.json()['response'])

In [42]:
# 3. RUN PIPELINE
image_path = "john_doe_eob.jpg"
print("Reading document...")
raw_text = extract_text_doctr(image_path)
print("Extracting fields...")
structured_data = extract_with_local_llm(raw_text)

Reading document...
Failed download. Trying https -> http instead. Downloading http://doctr-static.mindee.com/models?id=v0.7.0/db_resnet50-79bd7d70.pt&src=0 to /Users/michellefronda/.cache/doctr/models/db_resnet50-79bd7d70.pt


URLError: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1081)>

In [10]:
# Save to Pandas
df = pd.DataFrame([structured_data])
df.to_csv("local_medical_extracts.csv", index=False)
print("Done! Saved to CSV.")

NameError: name 'structured_data' is not defined